# Base model evaluation
In this notebook, the performance of selected base models will be evaluated on long texts.

**Selected models:**
- Encoder-only:
    - XLM-RoBERTa-large (FacebookAI/xlm-roberta-large) (0.6B parameters)
- Decoder-based:
    - Qwen3-Embedding-0.6B (Qwen/Qwen3-Embedding-0.6B) (0.6B parameters)

In [ ]:
!pip install mteb

In [ ]:
import numpy as np

import torch
from datasets import load_dataset
import pandas as pd

import datasets

## Baseline models

In [ ]:
from transformers import AutoTokenizer, AutoModel

In [ ]:
def tokenize_chunking_strategy(tokenizer, inputs, chunk_size, overlap):
    real_chunks_size = chunk_size - tokenizer.num_special_tokens_to_add(pair=False)    # for each chunk special tokens will be appened after

    number_of_chunks = []    # number of chunks for each text in input
    outer_chunked_texts_batch = []    # long batch of all texts chunks

    for text in inputs:
        token_ids = tokenizer(text, add_special_tokens=False, return_tensors="pt")["input_ids"].squeeze()
        start = 0
        chunk_number = 0
        while start < len(token_ids):
            end = start + real_chunks_size
            chunk_tokens = token_ids[start:end]
            chunk_text = tokenizer.decode(chunk_tokens, skip_special_tokens=True)
            outer_chunked_texts_batch.append(chunk_text)
            start += real_chunks_size - overlap
            chunk_number += 1
        number_of_chunks.append(chunk_number)

    tokenized_outer_batch = tokenizer(
        outer_chunked_texts_batch,
        add_special_tokens=True,
        max_length=chunk_size,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    )
    return (tokenized_outer_batch, number_of_chunks)

def re_group_chunked_outputs(outputs_last_hidden_state, number_of_chunks):
    # outputs shape: [ num_chunks * num_texts, chunk_size, *]
    # converting to [num_texts, num_chunks, chunk_size, *]
    re_grouped = []
    text_starts_i = 0
    for n_chunks in number_of_chunks:
        text_ends_i = text_starts_i + n_chunks
        re_grouped.append(
            outputs_last_hidden_state[text_starts_i:text_ends_i, :, :]
        )
        text_starts_i = text_ends_i
    return re_grouped

def tokenize_first_startegy(tokenizer, inputs, max_length):
    tokenized = tokenizer(
        inputs,
        add_special_tokens=True,
        padding="max_length",
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    )
    return tokenized


def tokenize_max_tokens_strategy(tokenizer, inputs):
    tokenized = tokenizer(
        inputs,
        add_special_tokens=True,
        padding="longest",
        truncation=True,
        max_length=tokenizer.model_max_length,
        return_tensors="pt")
    return tokenized

In [ ]:
from mteb import EncoderProtocol
from mteb.similarity_functions import cos_sim

In [ ]:
from enum import Enum

class Strategy(Enum):
    chunking = "chunking"
    first = "first"
    max_tokens = "max_tokens"

In [ ]:
class XMLRoBERTa(EncoderProtocol):

    name = "xlm-roberta-large"
    similarity = staticmethod(cos_sim)

    def __init__(self, max_size, strategy: Strategy, overlap):
        self.model = AutoModel.from_pretrained(
            "FacebookAI/xlm-roberta-large",
            dtype=torch.float16,
            low_cpu_mem_usage=True,
            device_map="auto")

        self.tokenizer = AutoTokenizer.from_pretrained("FacebookAI/xlm-roberta-large")

        self.max_size = max_size
        self.strategy = strategy
        self.overlap = overlap

        self.model.eval()

    def __encode_batch(
            self,
            texts,
            **kwargs) -> np.ndarray:

        if self.strategy == Strategy.chunking:
            tokenized, numbers_of_chunks = tokenize_chunking_strategy(self.tokenizer, texts, self.max_size, self.overlap)
        if self.strategy == Strategy.first:
            tokenized = tokenize_first_startegy(self.tokenizer, texts, self.max_size)
        if self.strategy == Strategy.max_tokens:
            tokenized = tokenize_max_tokens_strategy(self.tokenizer, texts)

        with torch.inference_mode():
            tokenized = {k: v.to(self.model.device) for k, v in tokenized.items()}
            outputs = self.model(**tokenized)

            if self.strategy == Strategy.chunking:
                re_grouped = re_group_chunked_outputs(outputs.last_hidden_state, numbers_of_chunks)
                embeddings = torch.stack([
                    t.mean(dim=1).mean(dim=0).detach().cpu()
                    for t in re_grouped
                ])
            if (self.strategy == Strategy.first
                or self.strategy == Strategy.max_tokens):
                embeddings = outputs.last_hidden_state.mean(dim=1)


        embeddings = embeddings.detach().cpu().numpy()

        del outputs, tokenized
        torch.cuda.empty_cache()
        return embeddings

    def encode(
            self,
            inputs,
            *,
            task_metadata,
            hf_split: str,
            hf_subset: str,
            prompt_type,
            **kwargs):

        batch_size = kwargs["batch_size"]

        texts = [text for batch in inputs for text in batch["text"]]

        all_embeddings = []

        for start in range(0, len(texts), batch_size):
            batch = texts[start:start + batch_size]
            batch_embeddings = self.__encode_batch(batch)
            all_embeddings.append(batch_embeddings)

        return_embeddings = np.vstack(all_embeddings)
        print(return_embeddings.shape)
        return return_embeddings



class Qwen3_Embedding(EncoderProtocol):

    name = "Qwen3-Embedding-0.6B"
    similarity = staticmethod(cos_sim)

    def __init__(self, max_size, strategy, overlap, return_hidden_states=False):
        self.tokenizer = AutoTokenizer.from_pretrained(
            "Qwen/Qwen3-Embedding-0.6B",
            padding_side='left')

        self.model = AutoModel.from_pretrained(
            "Qwen/Qwen3-Embedding-0.6B",
            dtype=torch.float16,
            low_cpu_mem_usage=True,
            device_map="auto")

        self.max_size = max_size
        self.strategy = strategy
        self.overlap = overlap

        self.return_hidden_states = return_hidden_states

        self.model.eval()

    def __get_eos_token_embedding(self, last_hidden_states):
        return last_hidden_states[:, -1]

    def preprocess_query(self, texts):
        task = 'Given a search query, retrieve relevant passages that answer the query'
        return f'Instruct: {task}\nQuery:{texts}'

    def __encode_batch(self, texts, **kwargs) -> np.ndarray:
        if self.strategy == Strategy.chunking:
            tokenized, numbers_of_chunks = tokenize_chunking_strategy(
                self.tokenizer, texts, self.max_size, self.overlap
            )
        elif self.strategy == Strategy.first:
            tokenized = tokenize_first_startegy(self.tokenizer, texts, self.max_size)
        else:
            tokenized = tokenize_max_tokens_strategy(self.tokenizer, texts)

        with torch.inference_mode():
            tokenized = {k: v.to(self.model.device) for k, v in tokenized.items()}
            outputs = self.model(**tokenized)

            if self.strategy == Strategy.chunking:
                re_grouped = re_group_chunked_outputs(
                    outputs.last_hidden_state, numbers_of_chunks
                )
                embeddings = torch.stack([
                    self.__get_eos_token_embedding(t).mean(dim=0)
                    for t in re_grouped
                ])
            else:
                embeddings = self.__get_eos_token_embedding(
                    outputs.last_hidden_state
                )

        embeddings = embeddings.detach().cpu().numpy()

        del outputs, tokenized
        torch.cuda.empty_cache()
        return embeddings

    def encode(
            self,
            inputs,
            *,
            task_metadata,
            hf_split: str,
            hf_subset: str,
            prompt_type,
            **kwargs):

        batch_size = kwargs["batch_size"]

        texts = [text for batch in inputs for text in batch["text"]]
        if prompt_type.value == "query":
            texts = [self.preprocess_query(t) for t in texts]

        all_embeddings = []

        for start in range(0, len(texts), batch_size):
            batch = texts[start:start + batch_size]
            batch_embeddings = self.__encode_batch(batch)
            all_embeddings.append(batch_embeddings)

        return_embeddings = np.vstack(all_embeddings)
        print(return_embeddings.shape)
        return return_embeddings

## LongEmbed LEMBWikimQARetrieval

In [ ]:
import mteb

In [ ]:
long_embed_wiki_task = mteb.get_task("LEMBWikimQARetrieval")

In [ ]:
import json
import gc

strategies = [
    {
        "name": "Chunking",
        "strategy": Strategy.chunking,
        "max_size": 512,
        "overlap": 0
    },
    {
        "name": "Chunking 64 Overlap",
        "strategy": Strategy.chunking,
        "max_size": 512,
        "overlap": 64
    },
    {
        "name": "First",
        "strategy": Strategy.first,
        "max_size": 512,
        "overlap": None
    },
]

main_scores_roberta = []
roberta = None

for strategy in strategies:
    roberta = XMLRoBERTa(
        max_size=strategy["max_size"],
        strategy=strategy["strategy"],
        overlap=strategy["overlap"])

    eval = long_embed_wiki_task.evaluate(roberta, encode_kwargs={"batch_size": 4})
    main_scores_roberta.append(eval["default"]["main_score"])
    # save scores, eval is dictionary
    file_name = f"roberta_{strategy['name']}_scores.json"
    with open(file_name, "w") as f:
        json.dump(eval, f)

    # free memory just in case
    del roberta
    torch.cuda.empty_cache()
    gc.collect()

main_scores_qwen3 = []
qwen3 = None

for strategy in strategies:
    qwen3 = Qwen3_Embedding(
        max_size=strategy["max_size"],
        strategy=strategy["strategy"],
        overlap=strategy["overlap"])

    eval = long_embed_wiki_task.evaluate(qwen3, encode_kwargs={"batch_size": 4})
    main_scores_qwen3.append(eval["default"]["main_score"])
    # save scores, eval is dictionary
    file_name = f"qwen3_{strategy['name']}_scores.json"
    with open(file_name, "w") as f:
        json.dump(eval, f)

    # free memory just in case
    del qwen3
    torch.cuda.empty_cache()
    gc.collect()

Filtering queries by qrels:   0%|          | 0/300 [00:00<?, ? examples/s]

Processing queries for dataloading:   0%|          | 0/300 [00:00<?, ? examples/s]

(300, 1024)


Converting corpus dict:   0%|          | 0/300 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (4234 > 512). Running this sequence through the model will result in indexing errors


...

Filtering queries by qrels:   0%|          | 0/300 [00:00<?, ? examples/s]

Processing queries for dataloading:   0%|          | 0/300 [00:00<?, ? examples/s]

(300, 1024)


Converting corpus dict:   0%|          | 0/300 [00:00<?, ? examples/s]

(300, 1024)


In [ ]:
main_scores_roberta

[0.01584, 0.01657, 0.01804]

In [ ]:
main_scores_qwen3

[0.76854, 0.76813, 0.56949]

In [ ]:
pd.DataFrame(
    {
        "Strategy": [s["name"] for s in strategies],
        "Qwen3 score": main_scores_qwen3,
        "XML-RoBERTa score": main_scores_roberta
    }
).set_index("Strategy")

,Qwen3 score,XML-RoBERTa score
Strategy,,
Chunking,0.76854,0.01584
Chunking 64 Overlap,0.76813,0.01657
First,0.56949,0.01804


# Memory module

In [26]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [108]:
class Writer(nn.Module):

    def __init__(self, hidden_state_dim):
        super().__init__()

        self.hidden_state_dim = hidden_state_dim

        # attention Key and Query matrix
        self.W_k = nn.Linear(self.hidden_state_dim, self.hidden_state_dim, bias=False)
        self.W_q = nn.Linear(self.hidden_state_dim, self.hidden_state_dim, bias=False)

        # GRU matrices
        self.W_r = nn.Linear(self.hidden_state_dim, self.hidden_state_dim)
        self.U_r = nn.Linear(self.hidden_state_dim, self.hidden_state_dim)
        self.W_z = nn.Linear(self.hidden_state_dim, self.hidden_state_dim)
        self.U_z = nn.Linear(self.hidden_state_dim, self.hidden_state_dim)
        self.W_c = nn.Linear(self.hidden_state_dim, self.hidden_state_dim)
        self.U_c = nn.Linear(self.hidden_state_dim, self.hidden_state_dim)

    def get_attention_weights(self, h, memory):
        """
        h: hidden state (batch_size, hidden_state_dim)
        memory: memory (batch_size, n_units, hidden_state_dim)
        """
        k = self.W_k(memory)    # (batch_size, n_units, hidden_state_dim)
        q = self.W_q(h).unsqueeze(-1)    # (batch_size, hidden_state_dim, 1)
        scores = torch.bmm(k, q).squeeze(-1)    # (batch_size, n_units)
        attention_weights = F.softmax(scores, dim=1)    # (batch_size, n_units)
        return attention_weights

    def forward(self, input, memory):
        """
        input: input (batch_size, hidden_state_dim)
        memory: memory (batch_size, n_units, hidden_state_dim)
        """
        # Attention
        attention_weights = self.get_attention_weights(input, memory)

        # GRU
        input_expanded = input.unsqueeze(1).expand_as(memory)
        r = F.sigmoid(self.W_r(input_expanded) + self.U_r(memory))
        z = F.sigmoid(self.W_z(input_expanded) + self.U_z(memory))

        m_hat = F.tanh(self.W_c(input_expanded) + self.U_c(r * memory))
        m = z * m_hat + (1 - z) * memory
        return m


class Reader(nn.Module):
    def __init__(self, hidden_state_dim):
        super().__init__()

        self.hidden_state_dim = hidden_state_dim

        # attention Key and Query matrix
        self.W_k = nn.Linear(self.hidden_state_dim, self.hidden_state_dim)
        self.W_q = nn.Linear(self.hidden_state_dim, self.hidden_state_dim)

        # GRU matrices
        self.W_r = nn.Linear(self.hidden_state_dim, self.hidden_state_dim)
        self.U_r = nn.Linear(self.hidden_state_dim, self.hidden_state_dim)
        self.W_z = nn.Linear(self.hidden_state_dim, self.hidden_state_dim)
        self.U_z = nn.Linear(self.hidden_state_dim, self.hidden_state_dim)
        self.W_c = nn.Linear(self.hidden_state_dim, self.hidden_state_dim)
        self.U_c = nn.Linear(self.hidden_state_dim, self.hidden_state_dim)

    def get_attention_weights(self, h, memory):
        """
        h: hidden state (batch_size, hidden_state_dim)
        memory: memory (batch_size, n_units, hidden_state_dim)
        """
        k = self.W_k(memory)    # (batch_size, n_units, hidden_state_dim)
        q = self.W_q(h).unsqueeze(-1)    # (batch_size, hidden_state_dim, 1)
        scores = torch.bmm(k, q).squeeze(-1)    # (batch_size, n_units)
        attention_weights = F.softmax(scores, dim=1)    # (batch_size, n_units)
        return attention_weights

    def forward(self, input, memory):
        # Attention
        attention_weights = self.get_attention_weights(input, memory)

        pooled_memory = torch.bmm(memory, attention_weights)

        # GRU
        r = torch.sigmoid(self.W_r(pooled_memory) + self.U_r(input))
        z = torch.sigmoid(self.W_z(pooled_memory) + self.U_z(input))

        h_hat = torch.tanh(self.W_c(pooled_memory) + self.U_c(r * input))
        h = z * h_hat + (1 - z) * input
        return h


class Memory(nn.Module):

    def __init__(self, hidden_state_dim, n_units, batch_size=1):
        super().__init__()
        self.hidden_state_dim = hidden_state_dim
        self.n_units = n_units
        self.batch_size = batch_size
        self.memory_cells = torch.zeros(batch_size, n_units, self.hidden_state_dim)

        # reader will be only helpful for the token generation task.
        # for the embedding generation, only the writer will be used essentially.
        self.reader = Reader(hidden_state_dim)
        self.writer = Writer(hidden_state_dim)

    def reset(self, n_units, batch_size):
        self.n_units = n_units
        self.batch_size = batch_size
        self.memory_cells = torch.zeros(self.batch_size, self.n_units, self.hidden_state_dim)
        self.memory_cells.to(self.writer.W_k.weight.device)

    def forward(self, input, write_only=True):
        self.memory_cells = self.writer(input, self.memory_cells)
        if write_only:
            output = self.reader(input, self.memory_cells)
            return output
        return self.memory_cells


class MemoryPooling(nn.Module):

    def __init__(self, memory_size, hidden_size):
        """
        Pooling using the single head attention
        """
        super().__init__()
        self.memory_size = memory_size
        self.hidden_size = hidden_size
        self.W_k = nn.Linear(hidden_size, hidden_size, bias=False)
        self.Q_k = nn.Linear(hidden_size, hidden_size, bias=False)

        self.query = nn.Parameter(torch.randn(hidden_size))    # learnable query vector

        # MLP
        self.ff = nn.Sequential(
            nn.Linear(hidden_size, hidden_size * 4),
            nn.GELU(),
            nn.Linear(hidden_size * 4, hidden_size),
        )

    def forward(self, memory):
        batch_size = memory.shape[0]
        q = self.query.unsqueeze(0).expand(batch_size, -1)  # (B, D)
        K = self.W_k(memory)  # (B, N, D)
        q = self.Q_k(q).unsqueeze(-1)  # (B, D, 1)

        scores = torch.bmm(K, q).squeeze(-1)  # (B, N)
        attention_scores = F.softmax(scores, dim=1)  # (B, N)

        pooled = torch.bmm(attention_scores.unsqueeze(1), memory).squeeze(1)  # (B, D)

        output = self.ff(pooled)
        return output


class TransformerWithMemory(nn.Module):

    def __init__(
            self,
            tokenizer,
            transformer,
            memory_size):
        super().__init__()
        self.tokenizer = tokenizer
        self.transformer = transformer
        # transformr object can return a hidden states for all tokens
        self.memory_size = memory_size

        self.memory_module = Memory(
            hidden_state_dim = self.transformer.hidden_size,
            n_units=memory_size,
            batch_size=1)

        self.memory_pool = MemoryPooling(
            memory_size,
            self.transformer.hidden_size)

        self.padding_value = torch.zeros(self.transformer.hidden_size, dtype=torch.float) + 1000

    def get_hidden_states_per_batch(self, last_hidden_state, batch_size, numbers_of_chunks):
        """
        last_hidden_state:
            (batch_size, numbers_of_chunks, chunk_size, hidden_size)
        """

        padded_hidden_states = []

        for i in range(batch_size):
            hidden_states_batch = last_hidden_state[i, :numbers_of_chunks[i], :]
            padded_hidden_states.append(hidden_states_batch)

        hidden_state_dim = self.transformer.hidden_size

        padded_hidden_states = nn.utils.rnn.pad_sequence(
            padded_hidden_states,
            batch_first=True,
            padding_value=self.padding_value
        )
        return padded_hidden_states

    def get_hidden_states(self, input):
        tokenized, numbers_of_chunks = tokenize_chunking_strategy(
            self.tokenizer,
            input,
            self.max_size,
            self.overlap)

        outputs = self.transformer(tokenized)

        hidden_states = self.get_hidden_states_per_batch(
            outputs.hidden_states,
            tokenized.size(0),
            numbers_of_chunks)
        return hidden_states

    def write_memory(self, input):
        """
        input: (batch_size, seq_len) - the hidden states from the transformer
        """
        batch_size = input.shape[0]
        self.memory_module.reset(
            n_units=self.memory_size,
            batch_size=batch_size)

        hidden_states = self.transformer(input).last_hidden_state  # (batch_size, seq_len, hidden_size)
        for i in range(len(hidden_states.shape[1])):
            # check if the hidden_state is not a padding
            if all(hidden_states == self.padding_value):
                break
            else:
                memory_cells = self.memory(hidden_states[:, i, :])

    def memory_pool(self):
        embed = self.memory_pooling(self.memory_module.memory_cells)
        return embed

    def encode(self, input):
        hidden_states = self.get_hidden_states(input)
        self.write_memory(hidden_states)
        return self.memory_pool()

In [109]:
# testing on the example data

memory_size = 3
batch_size = 2
hidden_size = 5

memory_module = Memory(hidden_size, memory_size, batch_size)
memory_pool = MemoryPooling(memory_size, hidden_size)

In [110]:
memory_module.reset(memory_size, batch_size)

In [101]:
memory = torch.randn((batch_size, memory_size, hidden_size))
h = torch.randn((batch_size, hidden_size))

write_return = memory_module.writer(h, memory)
write_return.shape == memory.shape

True

In [92]:
memory_pool = MemoryPooling(memory_size, hidden_size)
memory_pool(memory).shape

torch.Size([2, 5])

## Svaing and loading the memory parts


In [113]:
import torch
from pathlib import Path

def save_memory_parts(model, path: str):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            "memory_module": model.memory_module.state_dict(),
            "memory_pool": model.memory_pool.state_dict(),
        },
        path
    )

def load_memory_parts(model, path: str, device="cpu", strict=True):
    ckpt = torch.load(path, map_location=device)

    missing_mm, unexpected_mm = model.memory_module.load_state_dict(
        ckpt["memory_module"], strict=strict
    )
    missing_mp, unexpected_mp = model.memory_pool.load_state_dict(
        ckpt["memory_pool"], strict=strict
    )

    return {
        "missing_memory_module": missing_mm,
        "unexpected_memory_module": unexpected_mm,
        "missing_memory_pool": missing_mp,
        "unexpected_memory_pool": unexpected_mp,
    }

## TransformerWithMemory from the hugging face transformers

In [ ]:
from transformers import AutoTokenizer, AutoModel

def get_transformer_with_memory(
        model_name,
        chunk_size=512,
        memory_size=5):

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    taransformer = AutoModel.from_pretrained(
        model_name,
        dtype=torch.float16,
        low_cpu_mem_usage=True,
        device_map="auto")

    transformer_with_memory = TransformerWithMemory(
        tokenizer,
        taransformer,
        memory_size=memory_size)
    return transformer_with_memory

# Training

In [94]:
from datasets import load_dataset
from torch.utils.data import DataLoader

## MS MARCO dataset

In [95]:
class MSMARCOCollator:

    def __call__(self, batch):
        # batch_size is the number of rows in the batch
        # allign the batch as:
        # batch = {
        #      to_encode: (query1, doc_11, doc_12 ...
        #                  query2, doc_21, doc_22 ...
        #      sample_indexes: [],    # pairs of (start, end) of the sample
        #      pos_mask: []
        #      }

        batch_return = {
            "to_encode": [],
            "n_documents": [],
            "pos_mask": []
        }

        to_encode = []
        sample_indexes = []
        pos_mask = []

        query_index = 0
        for row in batch:
            to_encode.append(row["query"])
            to_encode.extend(row["passages"]["passage_text"])
            sample_indexes.append(query_index)
            n_documents = len(row["passages"]["passage_text"])
            # pairs of (sample_start, sample)
            sample_indexes.append((query_index, n_documents + query_index + 1))
            query_index = n_documents + query_index + 1
            pos_mask.append(row["passages"]["is_selected"])

        return {
            "to_encode": to_encode,
            "sample_indexes": sample_indexes,
            "pos_mask": pos_mask
        }

ms_marco_collator = MSMARCOCollator()

ms_marco_dataset = load_dataset("microsoft/ms_marco", "v1.1", split="train")
# Filter out rows with zero positives (otherwise loss becomes -inf)
ms_marco_dataset = ms_marco_dataset.filter(lambda ex: 1 in [int(x) for x in ex["passages"]["is_selected"]])

ms_marco_dataloader = DataLoader(
    ms_marco_dataset,
    batch_size=4,
    collate_fn=ms_marco_collator
)

README.md: 0.00B [00:00, ?B/s]

v1.1/validation-00000-of-00001.parquet:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

v1.1/train-00000-of-00001.parquet:   0%|          | 0.00/175M [00:00<?, ?B/s]

v1.1/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/10047 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/82326 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/9650 [00:00<?, ? examples/s]

Filter:   0%|          | 0/82326 [00:00<?, ? examples/s]

## Natural Questions dataset

In [96]:
class NaturalQuestionsCollator:

    def __call__(self, batch):

        to_encode = []
        sample_indexes = []
        query_index = 0
        for row in batch:
            to_encode.append(row["query"])
            to_encode.append(row["answer"])
            sample_indexes.append((query_index, query_index + 2))
            query_index += 2

        B = len(batch)
        pos_mask = torch.eye(B, dtype=torch.bool)

        return {
            "to_encode": to_encode,
            "sample_indexes": sample_indexes,
            "pos_mask": pos_mask
        }

natural_questions_collator = NaturalQuestionsCollator()

natural_questions_dataset = load_dataset(
    "sentence-transformers/natural-questions", split="train")

natural_questions_dataloader = DataLoader(
    natural_questions_dataset,
    batch_size=4,
    collate_fn=natural_questions_collator
)

README.md: 0.00B [00:00, ?B/s]

pair/train-00000-of-00001.parquet:   0%|          | 0.00/44.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/100231 [00:00<?, ? examples/s]

In [97]:
def get_query_doc_for_loss(batch, embeddings):
    sample_indexes = batch["sample_indexes"]
    pos_mask = batch["pos_mask"]
    q_embs = []
    p_embs = []
    pos_mask = []

    for i, (sample_start, sample_end) in enumerate(sample_indexes):
        q_embs.append(embeddings[sample_start])
        p_embs.append(embeddings[sample_start + 1:sample_end])
        pos_mask.append(torch.tensor(pos_mask[i]))
    return {
        "q_embs": q_embs,
        "p_embs": p_embs,
        "pos_mask": pos_mask
    }

In [98]:
def multi_positive_infonce_batch_multiple_docs(
        q_embs,
        p_embs,
        pos_mask,
        tau=0.05):
    """
    q_embs: (B, d)
    p_embs: list of length B, each (N_i, d)
    pos_mask: list of length B, each (N_i,) boolean
    """

    losses = []

    for i in range(len(q_embs)):
        q = q_embs[i]              # (d,)
        p = p_embs[i]              # (N_i, d)
        pos = pos_mask[i]          # (N_i,)

        logits = (q @ p.T) / tau   # (N_i,)
        log_probs = torch.log_softmax(logits, dim=0)

        loss_i = -log_probs[pos].mean()
        losses.append(loss_i)

    return torch.stack(losses).mean()


def multi_positive_infonce_in_batch_negative(
        q_emb,
        p_emb,
        pos_mask,
        tau=0.05):
    """
    q_emb: (B, d)
    p_emb: (B, d)
    pos_mask: (B, B) boolean
    """
    logits = (q_emb @ p_emb.T) / tau
    log_probs = torch.log_softmax(logits, dim=1)
    loss_per_query = -(log_probs * pos_mask).sum(dim=1) / pos_mask.sum(dim=1)
    return loss_per_query.mean()


def train(loss_fn, dataloader, model):
    device = "cuda" if torch.cuda.is_available() else "cpu"

    BATCH_SIZE = 4 if device == "cuda" else 2
    LR = 2e-5
    WEIGHT_DECAY = 0.01
    EPOCHS = 1

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    step = 0

    loss_data = []
    steps = []

    for epoch in range(EPOCHS):
        for batch in dataloader:

            # encode
            embeddings = model.encode(batch["to_encode"])

            embeddings_grouped = get_query_doc_for_loss(batch, embeddings)

            q_emb = embeddings_grouped["q_embs"]
            p_emb = embeddings_grouped["p_embs"]
            pos_mask = embeddings_grouped["pos_mask"]

            loss = loss_fn(q_emb, p_emb, pos_mask)

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            if step % 100 == 0:
                print(f"epoch={epoch} step={step} loss={loss.item():.4f}")
                loss_data.append(loss.item())
                steps.append(step)
            step += 1

    return {
        "losses": loss_data,
        "steps": step
    }